# Phase 2 - A2A Prompt Injection Detection Models (Training and Evaluation)

This notebook is **Phase 2** of the study *"Agent-to-Agent Prompt Injection in AI-Only Social Networks: A Taxonomy and Detection Study on Moltbook"*. Working on the **630-post seed set** produced in Phase 1 and labelled by hand (167 attacks / 463 non-attacks), it performs the following:

1. Template-based deduplication plus a **group-aware train/test split** (prevents template leakage).
2. Three detection approaches: **TF-IDF + logistic regression**, a fine-tuned **DistilBERT**, and a **zero-shot** (BART-MNLI) reference.
3. Threshold optimization, model comparison, **bootstrap 95% CI + McNemar** (statistical significance), **group-aware 5-fold cross-validation**, and a **temporal holdout** (train on phases 1+2, test on phase 3).
4. Error analysis, interpretability (most discriminative terms), confusion matrix, and category distribution.

**Input:** `moltbook_seed_v2.csv` (Phase 1 output, labelled by hand).

> **Note:** a GPU is recommended for DistilBERT (checked in Section 1). Run the cells in order.


## 0. Setup: Google Drive and folder structure

Uses the same folder as Phase 1; edit `PROJECT_DIR` to match your own directory.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# === EDIT THIS TO MATCH YOUR OWN FOLDER (same as Phase 1) ===
# Example: '/content/drive/MyDrive/Moltbook_Research'
PROJECT_DIR = '/content/drive/MyDrive/Moltbook_Research'
# ============================================================

DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')
FIG_DIR     = os.path.join(PROJECT_DIR, 'fig')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'output')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'output', 'model')

for d in [DATASET_DIR, FIG_DIR, OUTPUT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# Check that the Phase 1 output is present
SEED_CSV = os.path.join(DATASET_DIR, 'moltbook_seed_v2.csv')
assert os.path.exists(SEED_CSV), (
    f'Phase 1 output not found: {SEED_CSV}\n'
    'Please place moltbook_seed_v2.csv in the dataset/ folder.'
)
print('Drive ready, seed dataset found.')

## 1. Environment setup and GPU check

Installs the libraries and checks for a GPU for DistilBERT training.


In [ ]:
!pip install -q transformers datasets scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

sns.set_theme(style='whitegrid')

def save_fig(name, dpi=150):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Figure saved: {path}')

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU found. Select Runtime > Change runtime type > GPU.')

## 2. Data loading, deduplication and group-aware split

The labelled seed is loaded and `title` + `content` are merged into a single text field.


In [ ]:
# Read the label column as a string so it is not coerced to float
df = pd.read_csv(SEED_CSV, dtype={'label_is_attack': str})

# Clean the label and cast to int
df['label'] = df['label_is_attack'].str.strip().map({'1': 1, '0': 0})
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

# Text: title + content
df['text'] = (df['title'].fillna('') + ' . ' + df['content'].fillna('')).str.strip()

print(f'Total labelled examples: {len(df)}')
print(f"Attack (1):     {(df['label'] == 1).sum()}")
print(f"Not attack (0): {(df['label'] == 0).sum()}")
print(f'Shape: {df.shape}')

### 2.2 Template groups (deduplication)

Attacks are frequently near-duplicate, template-driven posts. Each post is assigned a **template signature** (`template_sig`) built from the first 80 characters of its normalized text, so variants of the same template fall into the same group.


In [ ]:
import re

def template_signature(text):
    norm = re.sub(r'http\S+', '', str(text).lower())
    norm = re.sub(r'[^a-z\s]', '', norm)
    norm = re.sub(r'\s+', ' ', norm).strip()
    return norm[:80]

df['template_sig'] = df['text'].apply(template_signature)

n_unique_groups = df['template_sig'].nunique()
print(f'Total posts: {len(df)}')
print(f'Unique template groups: {n_unique_groups}')

group_sizes = df['template_sig'].value_counts()
print('\nTen largest groups:')
print(group_sizes.head(10))

### 2.3 Group-aware train/test split

The split is made with `GroupShuffleSplit`; **no template group ever appears in both train and test**, which prevents the optimistic metrics that template leakage would produce (487 train / 143 test).


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df['text']
y = df['label']
groups = df['template_sig']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

n_train_attack = df_train['label'].sum()
n_test_attack = df_test['label'].sum()

print(f'Train: {len(df_train)} posts, {n_train_attack} attacks')
print(f'Test:  {len(df_test)} posts, {n_test_attack} attacks')

train_groups = set(df_train['template_sig'])
test_groups = set(df_test['template_sig'])
overlap = train_groups & test_groups
print(f'Train/test group overlap: {len(overlap)} (must be 0)')
assert len(overlap) == 0, 'LEAKAGE DETECTED'
print('No leakage; the split is safe.')

## 3. Baseline 1 - TF-IDF + logistic regression

An interpretable reference model: 1-2 gram TF-IDF with class-balanced logistic regression. The decision threshold is optimized to maximize F1 on the precision-recall curve.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, f1_score, precision_recall_curve)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(df_train['text'])
X_test_tfidf = vectorizer.transform(df_test['text'])

y_train = df_train['label'].values
y_test = df_test['label'].values

clf = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf.fit(X_train_tfidf, y_train)

y_proba = clf.predict_proba(X_test_tfidf)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
f1_default = f1_score(y_test, y_pred_default)

print(f'ROC-AUC: {roc_auc:.3f}')
print(f'PR-AUC:  {pr_auc:.3f}')
print(f'F1 (threshold=0.50): {f1_default:.3f}')
print()
print(classification_report(y_test, y_pred_default, target_names=['Not Attack', 'Attack']))

# Threshold optimization
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-10)
best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f'Optimal threshold: {best_threshold:.3f}')
print(f'Optimal F1:        {best_f1:.3f}')
print(f'Precision:         {precisions[best_idx]:.3f}')
print(f'Recall:            {recalls[best_idx]:.3f}')

## 4. Main model - DistilBERT fine-tuning

The fine-tuned transformer. The cells below cover, in order: the tokenizer and `Dataset` class, model loading, training with `Trainer` (max_len=256, batch=16, lr=2e-5, 4 epochs), and test-set predictions (`probs`).


In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class MoltbookDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=max_len
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = MoltbookDataset(df_train['text'], df_train['label'], tokenizer)
test_dataset = MoltbookDataset(df_test['text'], df_test['label'], tokenizer)

print(f'Training examples: {len(train_dataset)}')
print(f'Test examples:     {len(test_dataset)}')

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

n_neg = (df_train['label'] == 0).sum()
n_pos = (df_train['label'] == 1).sum()
weight_neg = len(df_train) / (2 * n_neg)
weight_pos = len(df_train) / (2 * n_pos)
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float).to(device)

print(f'Class weights: negative={weight_neg:.3f}, positive={weight_pos:.3f}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='/tmp/distilbert_v2',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    learning_rate=2e-5,
    seed=42,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

y_test_bert = df_test['label'].values

roc_auc_bert = roc_auc_score(y_test_bert, probs)
print(f'DistilBERT ROC-AUC: {roc_auc_bert:.3f}')

y_pred_default_bert = (probs >= 0.5).astype(int)
f1_default_bert = f1_score(y_test_bert, y_pred_default_bert)
print(f'F1 (threshold=0.50): {f1_default_bert:.3f}')
print()
print(classification_report(y_test_bert, y_pred_default_bert, target_names=['Not Attack', 'Attack']))

precisions_b, recalls_b, thresholds_b = precision_recall_curve(y_test_bert, probs)
f1_scores_b = 2 * precisions_b * recalls_b / (precisions_b + recalls_b + 1e-10)
best_idx_b = f1_scores_b[:-1].argmax()
best_threshold_b = thresholds_b[best_idx_b]
best_f1_b = f1_scores_b[best_idx_b]

print(f'Optimal threshold: {best_threshold_b:.3f}')
print(f'Optimal F1:        {best_f1_b:.3f}')
print(f'Precision:         {precisions_b[best_idx_b]:.3f}')
print(f'Recall:            {recalls_b[best_idx_b]:.3f}')

In [ ]:
# Keep the DistilBERT probabilities under a separate name for later comparisons
probs_distilbert_global = probs.copy()

## 5. Baseline 2 - zero-shot transformer (untrained reference)

Zero-shot classification with BART-MNLI, which measures how well an off-the-shelf model with no task-specific training performs here (`zero_shot_probs`).


In [ ]:
from transformers import pipeline

zero_shot = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0 if device == 'cuda' else -1
)

candidate_labels = ['malicious prompt injection attack', 'normal harmless social media post']

# Evaluate the whole test set (143 examples; this may take a while)
zero_shot_probs = []
for text in df_test['text']:
    result = zero_shot(text[:512], candidate_labels)
    idx = result['labels'].index('malicious prompt injection attack')
    zero_shot_probs.append(result['scores'][idx])

zero_shot_probs = np.array(zero_shot_probs)
roc_auc_zs = roc_auc_score(y_test_bert, zero_shot_probs)
print(f'Zero-shot ROC-AUC: {roc_auc_zs:.3f}')

y_pred_zs = (zero_shot_probs >= 0.5).astype(int)
f1_zs = f1_score(y_test_bert, y_pred_zs)
print(f'F1 (threshold=0.50): {f1_zs:.3f}')
print()
print(classification_report(y_test_bert, y_pred_zs, target_names=['Not Attack', 'Attack']))

## 6. Model comparison and statistical significance

Performance of the three models on the held-out test set (n=143): ROC and PR curves, threshold-F1 curves, and a comparison table, all computed from the models' **actual probability outputs**.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
probs_distilbert_global = probs.copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ---------- Left panel: ROC ----------
for name, probs_model, color in [
    ('TF-IDF + LogReg', y_proba, '#6c8ebf'),
    ('DistilBERT', probs_distilbert_global, '#82b366'),
    ('Zero-shot (BART-MNLI)', zero_shot_probs, '#b85450'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs_model)
    auc_val = roc_auc_score(y_test, probs_model)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

axes[0].plot([0, 1], [0, 1], ls='--', color='gray', alpha=0.6)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('(a) ROC Curves')
axes[0].legend(fontsize=9, loc='lower right')

# ---------- Right panel: precision-recall ----------
for name, probs_model, color in [
    ('TF-IDF + LogReg', y_proba, '#6c8ebf'),
    ('DistilBERT', probs_distilbert_global, '#82b366'),
    ('Zero-shot (BART-MNLI)', zero_shot_probs, '#b85450'),
]:
    p, r, _ = precision_recall_curve(y_test, probs_model)
    pr_auc_val = average_precision_score(y_test, probs_model)
    axes[1].plot(r, p, label=f'{name} (AP={pr_auc_val:.3f})', color=color, linewidth=2)

baseline = y_test.sum() / len(y_test)
axes[1].axhline(baseline, ls='--', color='gray', alpha=0.6, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('(b) Precision-Recall Curves')
axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
save_fig('06_roc_pr_comparison.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name, probs_model, color in [
    (axes[0], 'TF-IDF + LogReg', y_proba, '#6c8ebf'),
    (axes[1], 'DistilBERT', probs_distilbert_global, '#82b366'),
]:
    p, r, t = precision_recall_curve(y_test, probs_model)
    f1_vals = 2 * p * r / (p + r + 1e-10)

    ax.plot(t, p[:-1], label='Precision', color='#9673a6', linewidth=1.8)
    ax.plot(t, r[:-1], label='Recall', color='#d79b00', linewidth=1.8)
    ax.plot(t, f1_vals[:-1], label='F1', color=color, linewidth=2.2)

    best_i = f1_vals[:-1].argmax()
    best_t = t[best_i]
    ax.axvline(best_t, ls='--', color='gray', alpha=0.7,
               label=f'Optimal threshold ({best_t:.3f})')

    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(name)
    ax.legend(fontsize=8.5, loc='lower left')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

plt.tight_layout()
save_fig('07_threshold_f1_curve.png')
plt.show()

In [ ]:
# Final model comparison table, formatted as LaTeX
roc_distilbert = roc_auc_score(y_test, probs_distilbert_global)
pr_distilbert = average_precision_score(y_test, probs_distilbert_global)
f1_distilbert_default = f1_score(y_test, (probs_distilbert_global >= 0.5).astype(int))

roc_zs = roc_auc_score(y_test, zero_shot_probs)
pr_zs = average_precision_score(y_test, zero_shot_probs)
f1_zs_default = f1_score(y_test, (zero_shot_probs >= 0.5).astype(int))

latex_table = r"""
\begin{table}[h]
\centering
\caption{Model performance comparison on the held-out test set (n=143).}
\label{tab:results}
\begin{tabular}{lccccc}
\toprule
Model & ROC-AUC & PR-AUC & F1 (0.50) & F1 (optimal) & Optimal threshold \\
\midrule
""" + f"""TF-IDF + LogReg & {roc_auc:.3f} & {pr_auc:.3f} & {f1_default:.3f} & {best_f1:.3f} & {best_threshold:.3f} \\\\
DistilBERT (fine-tuned) & {roc_distilbert:.3f} & {pr_distilbert:.3f} & {f1_distilbert_default:.3f} & {best_f1_b:.3f} & {best_threshold_b:.3f} \\\\
Zero-shot (BART-MNLI) & {roc_zs:.3f} & {pr_zs:.3f} & {f1_zs_default:.3f} & --- & --- \\\\
""" + r"""\bottomrule
\end{tabular}
\end{table}
"""

print(latex_table)

### 6.2 Bootstrap 95% CI + McNemar (significance testing)

ROC-AUC and F1 confidence intervals from 5,000 resamples; a paired F1-difference CI and a McNemar test for TF-IDF vs DistilBERT. *(Result: the difference between the two trained models is not significant, while both clearly beat zero-shot.)*


In [ ]:
# --- Guard: are the required variables in memory? ---
import numpy as np
_req = ['y_test', 'y_proba', 'probs_distilbert_global', 'zero_shot_probs']
_missing = [v for v in _req if v not in dir()]
assert not _missing, f'Missing variables; run the model cells first: {_missing}'
_lens = {v: len(eval(v)) for v in _req}
assert len(set(_lens.values())) == 1, f'Arrays are not aligned to the same test set: {_lens}'
print('Guard OK - all arrays present and of equal length:', _lens)

In [ ]:
# === Bootstrap 95% CI + paired model comparison (McNemar) ===
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score
from scipy.stats import binomtest

yt = np.asarray(y_test)
models = {'TF-IDF+LogReg': np.asarray(y_proba),
          'DistilBERT':    np.asarray(probs_distilbert_global),
          'Zero-shot':     np.asarray(zero_shot_probs)}

def opt_thr(y, s):
    p, r, t = precision_recall_curve(y, s)
    f1 = 2 * p * r / (p + r + 1e-12)
    i = np.nanargmax(f1[:-1])
    return float(t[i])

thr = {m: opt_thr(yt, s) for m, s in models.items()}

# --- Bootstrap CI (5,000 resamples, threshold held fixed) ---
rng = np.random.default_rng(42); B = 5000; n = len(yt)

def boot(s, threshold):
    aucs, f1s = [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(np.unique(yt[idx])) < 2:  # skip resamples with a single class
            continue
        aucs.append(roc_auc_score(yt[idx], s[idx]))
        f1s.append(f1_score(yt[idx], (s[idx] >= threshold).astype(int), zero_division=0))
    ci = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))
    return (roc_auc_score(yt, s), ci(aucs),
            f1_score(yt, (s >= threshold).astype(int), zero_division=0), ci(f1s))

print('=== Bootstrap 95% CI (threshold = optimal) ===')
for m, s in models.items():
    auc, aci, f1, fci = boot(s, thr[m])
    print(f'{m:16s} ROC-AUC={auc:.3f} [{aci[0]:.3f}, {aci[1]:.3f}] | '
          f'F1(opt={thr[m]:.3f})={f1:.3f} [{fci[0]:.3f}, {fci[1]:.3f}]')

# --- Paired difference: TF-IDF minus DistilBERT (same resamples) ---
sa, sb = models['TF-IDF+LogReg'], models['DistilBERT']
ta, tb = thr['TF-IDF+LogReg'], thr['DistilBERT']
diffs = []
for _ in range(B):
    idx = rng.integers(0, n, n)
    if len(np.unique(yt[idx])) < 2:
        continue
    fa = f1_score(yt[idx], (sa[idx] >= ta).astype(int), zero_division=0)
    fb = f1_score(yt[idx], (sb[idx] >= tb).astype(int), zero_division=0)
    diffs.append(fa - fb)
dlo, dhi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
print(f'\nF1 difference (TF-IDF minus DistilBERT): {np.mean(diffs):+.3f} '
      f'(95% CI: [{dlo:+.3f}, {dhi:+.3f}])')
print('-> If the CI contains 0, the difference is NOT statistically significant.')

# --- McNemar (paired predictions at the optimal thresholds) ---
pa = (sa >= ta).astype(int); pb = (sb >= tb).astype(int)
a_ok, b_ok = (pa == yt), (pb == yt)
n01 = int(np.sum(a_ok & ~b_ok))   # TF-IDF correct, DistilBERT wrong
n10 = int(np.sum(~a_ok & b_ok))   # TF-IDF wrong, DistilBERT correct
pval = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if (n01 + n10) > 0 else 1.0
print(f'\nMcNemar: TF-IDF correct / DistilBERT wrong = {n01}, '
      f'TF-IDF wrong / DistilBERT correct = {n10}, exact p = {pval:.3f}')
print('-> If p > 0.05 there is no significant difference in accuracy between the two models.')

### 6.3 Temporal generalizability (holdout)

Trains on phases 1+2 and tests on phase 3 (group-clean), measuring whether the model generalizes over time under category drift.


In [ ]:
# === Temporal generalizability: train on phases 1+2, test on phase 3 ===
import re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, f1_score, classification_report)

d = df.copy()
if 'text' not in d:
    d['text'] = (d['title'].fillna('') + ' ' + d['content'].fillna(''))
if 'label' not in d:
    d['label'] = d['label_is_attack'].astype(int)
if 'template_sig' not in d:
    def tsig(x):
        x = re.sub(r'http\S+', '', str(x).lower())
        x = re.sub(r'[^a-z\s]', '', x)
        return re.sub(r'\s+', ' ', x).strip()[:80]
    d['template_sig'] = d['text'].apply(tsig)

is_test = d['phase'].eq('Phase 3').values
train_groups = set(d.loc[~is_test, 'template_sig'])
keep_test = is_test & ~d['template_sig'].isin(train_groups).values   # prevent group leakage

d_tr, d_te = d[~is_test], d[keep_test]
print(f"Train (phases 1+2): {len(d_tr)} posts, {int(d_tr['label'].sum())} attacks")
print(f"Test  (phase 3, group-clean): {len(d_te)} posts, {int(d_te['label'].sum())} attacks "
      f"({int(is_test.sum()) - len(d_te)} posts removed because of group overlap)")
if d_te['label'].nunique() < 2:
    print('WARNING: the phase 3 test set contains a single class; the sample may be too small.')

vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
Xtr, Xte = vec.fit_transform(d_tr['text']), vec.transform(d_te['text'])
ytr, yte = d_tr['label'].values, d_te['label'].values
clf_temporal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(Xtr, ytr)
proba = clf_temporal.predict_proba(Xte)[:, 1]

p, r, t = precision_recall_curve(yte, proba)
f1c = 2 * p * r / (p + r + 1e-12)
bi = np.nanargmax(f1c[:-1]); bt = t[bi]
print('\n--- Temporal holdout results (phase 3) ---')
print(f'ROC-AUC = {roc_auc_score(yte, proba):.3f} | PR-AUC = {average_precision_score(yte, proba):.3f}')
print(f'F1(threshold=0.50) = {f1_score(yte, (proba >= 0.5).astype(int), zero_division=0):.3f}')
print(f'F1(optimal={bt:.3f}) = {f1c[bi]:.3f}')
print(f'\n(For comparison, the random group-aware split above: '
      f'ROC-AUC={roc_auc:.3f}, F1(opt)={best_f1:.3f})')
print(classification_report(yte, (proba >= bt).astype(int),
                            target_names=['Not Attack', 'Attack'], zero_division=0))

## 7. Group-aware cross-validation

To counter the optimism of a single split, a 5-fold group-aware cross-validation in which template groups never cross fold boundaries (ROC-AUC and optimal F1, mean +/- std).


In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

X_all = df['text']
y_all = df['label'].values
groups_all = df['template_sig']

gkf = GroupKFold(n_splits=5)

roc_aucs = []
f1s_default = []
f1s_optimal = []
fold_results = []

for fold, (train_i, test_i) in enumerate(gkf.split(X_all, y_all, groups=groups_all)):
    X_tr, X_te = X_all.iloc[train_i], X_all.iloc[test_i]
    y_tr, y_te = y_all[train_i], y_all[test_i]

    g_tr = set(groups_all.iloc[train_i])
    g_te = set(groups_all.iloc[test_i])
    assert len(g_tr & g_te) == 0, f'Fold {fold}: LEAKAGE DETECTED'

    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
    X_tr_vec = vec.fit_transform(X_tr)
    X_te_vec = vec.transform(X_te)

    clf_cv = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    clf_cv.fit(X_tr_vec, y_tr)

    proba_cv = clf_cv.predict_proba(X_te_vec)[:, 1]

    if len(np.unique(y_te)) > 1:
        roc = roc_auc_score(y_te, proba_cv)
        roc_aucs.append(roc)
    else:
        roc = float('nan')

    pred_default = (proba_cv >= 0.5).astype(int)
    f1_def = f1_score(y_te, pred_default, zero_division=0)
    f1s_default.append(f1_def)

    p, r, t = precision_recall_curve(y_te, proba_cv)
    f1_curve = 2 * p * r / (p + r + 1e-10)
    best_f1_cv = f1_curve[:-1].max() if len(f1_curve) > 1 else 0
    f1s_optimal.append(best_f1_cv)

    fold_results.append((fold + 1, len(test_i), int(y_te.sum()), roc, f1_def, best_f1_cv))

    print(f'Fold {fold+1}: n_test={len(test_i)}, positives={y_te.sum()}, '
          f'ROC-AUC={roc:.3f}, F1(0.5)={f1_def:.3f}, F1(opt)={best_f1_cv:.3f}')

print()
print(f'ROC-AUC: {np.mean(roc_aucs):.3f} +/- {np.std(roc_aucs):.3f}')
print(f'F1 (default threshold): {np.mean(f1s_default):.3f} +/- {np.std(f1s_default):.3f}')
print(f'F1 (optimal threshold): {np.mean(f1s_optimal):.3f} +/- {np.std(f1s_optimal):.3f}')

In [ ]:
# Cross-validation table in LaTeX form (values taken from the loop above)
mean_roc = np.mean(roc_aucs)
std_roc = np.std(roc_aucs)
mean_f1_def = np.mean(f1s_default)
std_f1_def = np.std(f1s_default)
mean_f1_opt = np.mean(f1s_optimal)
std_f1_opt = np.std(f1s_optimal)

latex_cv_table = r"""
\begin{table}[h]
\centering
\caption{5-fold group-based cross-validation results (TF-IDF + Logistic Regression).}
\label{tab:cv}
\begin{tabular}{lcccc}
\toprule
Fold & Test size & Positives & ROC-AUC & F1 (optimal) \\
\midrule
"""
for f, n_te, pos, roc, f1d, f1o in fold_results:
    latex_cv_table += f'{f} & {n_te} & {pos} & {roc:.3f} & {f1o:.3f} \\\\\n'

latex_cv_table += r'\midrule' + '\n'
latex_cv_table += (f'\\textbf{{Mean}} & --- & --- & '
                   f'\\textbf{{{mean_roc:.3f} $\\pm$ {std_roc:.3f}}} & '
                   f'\\textbf{{{mean_f1_opt:.3f} $\\pm$ {std_f1_opt:.3f}}} \\\\\n')
latex_cv_table += r"""\bottomrule
\end{tabular}
\end{table}
"""

print(latex_cv_table)

## 8. Error analysis and interpretability

Posts misclassified by the best model (confusion matrix), the most discriminative terms, and the category distribution of the labelled set.


In [ ]:
y_pred_opt_final = (y_proba >= best_threshold).astype(int)

df_test_analysis = df_test.copy()
df_test_analysis['proba'] = y_proba
df_test_analysis['pred'] = y_pred_opt_final
df_test_analysis['true'] = y_test

false_negatives = df_test_analysis[(df_test_analysis['true'] == 1) & (df_test_analysis['pred'] == 0)]
false_positives = df_test_analysis[(df_test_analysis['true'] == 0) & (df_test_analysis['pred'] == 1)]

print(f'False negatives (missed attacks): {len(false_negatives)}')
print(f'False positives (false alarms):   {len(false_positives)}')
print()
print('=== FALSE NEGATIVES ===')
for i, r in false_negatives.iterrows():
    print(f"id={r['id'][:8]} | proba={r['proba']:.3f} | category={r['label_category']} | {r['title'][:70]}")
print()
print('=== FALSE POSITIVES ===')
for i, r in false_positives.iterrows():
    print(f"id={r['id'][:8]} | proba={r['proba']:.3f} | heuristic={r['heuristic_categories']} | {r['title'][:70]}")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_opt_final)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')

labels = ['Not Attack', 'Attack']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (threshold={best_threshold:.3f})')

for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color=color, fontsize=16, fontweight='bold')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_fig('05_confusion_matrix.png')
plt.show()

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

coef_df = pd.DataFrame({'term': feature_names, 'coef': coefs})
coef_df_sorted = coef_df.sort_values('coef', ascending=False)

top_attack = coef_df_sorted.head(15)
top_safe = coef_df_sorted.tail(15).sort_values('coef')

print('=== Terms weighted most strongly towards ATTACK ===')
print(top_attack.to_string(index=False))
print()
print('=== Terms weighted most strongly towards NOT_ATTACK ===')
print(top_safe.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

combined = pd.concat([top_attack, top_safe])
combined = combined.sort_values('coef')

colors = ['#b85450' if c < 0 else '#6c8ebf' for c in combined['coef']]
ax.barh(combined['term'], combined['coef'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Most Discriminative Terms: Attack vs. Non-Attack')
ax.text(0.02, 0.98, 'Attack-indicative', transform=ax.transAxes,
        ha='left', va='top', color='#6c8ebf', fontweight='bold')
ax.text(0.98, 0.02, 'Non-attack-indicative', transform=ax.transAxes,
        ha='right', va='bottom', color='#b85450', fontweight='bold')

plt.tight_layout()
save_fig('08_discriminative_terms.png')
plt.show()

In [ ]:
from collections import Counter

# Count taxonomy categories over the positive posts (a post may carry several, e.g. 'C2,C5')
cat_counter = Counter()
for value in df.loc[df['label'] == 1, 'label_category'].fillna(''):
    for part in str(value).split(','):
        part = part.strip()
        if part:
            cat_counter[part] += 1

print('Category counts:', dict(sorted(cat_counter.items())))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Panel 1: category distribution (slanted x-axis labels)
cat_order = ['C1', 'C2', 'C3', 'C4', 'C5']
cat_labels = ['C1: Instruction Override', 'C2: Identity Hijacking',
              'C3: Advertising/Redirection', 'C4: Info Exfiltration',
              'C5: Memory Poisoning']
cat_counts = [cat_counter.get(c, 0) for c in cat_order]

x_pos = np.arange(len(cat_order))
axes[0].bar(x_pos, cat_counts, color='#6c8ebf')
for i, v in enumerate(cat_counts):
    axes[0].text(i, v + 1.5, str(v), ha='center', fontweight='bold')
axes[0].set_title('(a) Attack Category Distribution')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, max(cat_counts) * 1.15)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(cat_labels, rotation=30, ha='right')

# Panel 2: class balance pie chart
n_not_attack = int((df['label'] == 0).sum())
n_attack = int((df['label'] == 1).sum())
colors_pie = ['#82b366', '#b85450']
labels_pie = [f'Not Attack\n({n_not_attack})', f'Attack\n({n_attack})']

axes[1].pie(
    [n_not_attack, n_attack],
    labels=labels_pie,
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontweight': 'bold'}
)
axes[1].set_title(f'(b) Class Balance (n={len(df)})')

plt.tight_layout()
save_fig('09_dataset_overview.png')
plt.show()